<div dir="rtl">

## بخش ۰: پیش‌پردازش داده‌ها و کنترل استقلال مشاهدات
**پوشش‌دهنده: الزام ج.۵ (کنترل مراجعه‌های تکراری) و اصلاح پیش‌پردازش فاز اول**

قبل از شروع محاسبات احتمالی، داده‌های بالینی (`UDS`) و تصویربرداری (`MRI`) را ادغام و تمیزکاری می‌کنیم:

* **کنترل مراجعات تکراری (الزام ج.۵):** بررسی ستون `NACCID` نشان می‌دهد که در هر دو فایل دقیقاً ۱۲۳۷ بیمار یکتا وجود دارد و برای هر فرد فقط یک ویزیت ثبت شده است؛ بنابراین شرط استقلال مشاهدات کاملاً برقرار است.
* **اصلاح مقادیر اعشاری و پرت (طبق راهنمای NACC و نظر TA):** به دلیل اجرای الگوریتم جایگذاری (Imputation) روی دیتاست اولیه، برخی کدهای گمشده (مثل `9`، `88` و `888`) با اعداد واقعی میانگین گرفته شده‌اند. در این فاز به جای گرد کردن مقادیر اعشاری در متغیر اسمی `HYPERTEN`، فقط کدهای معتبر `0` (بدون سابقه)، `1` (فعال) و `2` (قدیمی) را نگه داشتیم و مقادیر اعشاری را به عنوان `NaN` حذف کردیم. همچنین مقادیر خارج از بازه مجاز پزشکی در متغیرهای کمی (`NACCBMI > 100`، `BPSYS > 230`، `MOCATOTS > 30` و `EDUC > 36`) فیلتر شدند تا در نهایت **۱۰۲۲ بیمار با داده‌های کامل و معتبر** باقی بمانند.

</div>

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns

uds_df = pd.read_csv('../data/udslong_imputed.csv')
mri_df = pd.read_csv('../data/mriscanclean.csv')

selected_uds = [
    'NACCID',
    'NACCAGE',
    'SEX',
    'EDUC',
    'NACCBMI',
    'BPSYS',
    'HYPERTEN',
    'MOCATOTS',
]
selected_mri = [
    'NACCID',
    'LH_SUPERIORFRONTAL_GVOL',
    'RH_SUPERIORFRONTAL_GVOL',
    'LH_SUPERIORFRONTAL_AVGTH',
    'RH_SUPERIORFRONTAL_AVGTH',
]

df_merged = pd.merge(
    uds_df[selected_uds], mri_df[selected_mri], on='NACCID', how='inner'
)

print(
    f'تعداد کل سطرها: {len(df_merged)} | تعداد بیماران یکتا:'
    f" {df_merged['NACCID'].nunique()}"
)

df_clean = df_merged.copy()
df_clean.loc[df_clean['EDUC'] > 36, 'EDUC'] = np.nan
df_clean.loc[
    (df_clean['NACCBMI'] > 100) | (df_clean['NACCBMI'] < 10), 'NACCBMI'
] = np.nan
df_clean.loc[(df_clean['BPSYS'] > 230) | (df_clean['BPSYS'] < 70), 'BPSYS'] = (
    np.nan
)
df_clean.loc[
    (df_clean['MOCATOTS'] > 30) | (df_clean['MOCATOTS'] < 0), 'MOCATOTS'
] = np.nan
df_clean.loc[~df_clean['HYPERTEN'].isin([0.0, 1.0, 2.0]), 'HYPERTEN'] = np.nan

df_final = df_clean.dropna().reset_index(drop=True)
df_final['HYPERTEN'] = df_final['HYPERTEN'].astype(int)
df_final['SEX'] = df_final['SEX'].astype(int)

print(f'تعداد نهایی بیماران پس از پاکسازی کامل: {len(df_final)}')

تعداد کل سطرها: 1237 | تعداد بیماران یکتا: 1237
تعداد نهایی بیماران پس از پاکسازی کامل: 1022
